# 🧠 Jour 3 — Le Perceptron de Rosenblatt : Théorie, Géométrie et Implémentation NumPy

Bienvenue dans ce troisième module consacré aux **fondations du Machine Learning et de l'IA** !

Après avoir maîtrisé le calcul vectoriel dans **Tool 1** et les matrices 2D, le broadcasting et la génération aléatoire dans **Tool 2**, nous franchissons une étape historique : construire de toutes pièces notre premier **algorithme d'apprentissage automatique** en pur **NumPy**.

Inventé en 1957 par **Frank Rosenblatt**, le **Perceptron** est le modèle formel qui a posé les bases de tous les réseaux de neurones artificiels modernes (des réseaux multicouches aux Transformers comme ChatGPT).

---

### 🎯 Objectifs de ce notebook :
1. **Comprendre l'anatomie d'un neurone artificiel** : de l'intuition biologique à la formule mathématique (somme pondérée + biais + fonction d'activation).
2. **Visualiser la géométrie de la décision** : pourquoi le Perceptron définit une droite (un hyperplan séparateur) dans l'espace des données.
3. **Maîtriser la règle d'apprentissage de Rosenblatt** : comment les poids s'ajustent automatiquement à chaque erreur ($y - \hat{y}$).
4. **Coder une classe `PerceptronNumPy` complète** : propagation avant vectorisée avec l'opérateur matriciel `@`, boucle d'apprentissage et arrêt anticipé.
5. **Résoudre les portes logiques (AND, OR)** et visualiser leurs frontières de séparation avec Matplotlib.
6. **Le drame du XOR et la séparabilité linéaire** : pourquoi le XOR est impossible pour un neurone unique, le premier *Hiver de l'IA* (1969), et la nécessité des réseaux multicouches (MLP).
7. **Grand Cas Pratique** : générer un jeu de données 2D synthétique avec NumPy, entraîner le Perceptron et tracer la frontière de décision ainsi que la courbe de convergence.

> 💡 **Pédagogie interactive** : Ce notebook contient des **QCM interactifs** et des cellules de vérification automatique `check()` pour valider vos réponses en direct.  
> 👉 `Shift + Entrée` pour exécuter chaque cellule pas à pas.


In [7]:
# Programme 1 — Initialisation & Outils interactifs (QCM et validation)
import numpy as np
import matplotlib.pyplot as plt
import random as _random_qcm
import ipywidgets as w
from IPython.display import display

# Relances humoristiques adaptées au Perceptron et au Machine Learning
_RELANCES_HUMOUR = [
    "Attention au biais : ton neurone penche du mauvais côté !",
    "Ce n'est pas un problème d'époques, mais de séparabilité linéaire !",
    "Erreur de dimension dans le produit matriciel X @ w !",
    "Même Frank Rosenblatt a dû réajuster ses poids synaptiques un jour.",
    "Aïe, le neurone a prédit 0 au lieu de 1 ! Augmente la somme pondérée !",
    "BroadcastError : tes features et tes poids ne parlent pas la même langue."
]

def qcm(question, options, correct, explication=""):
    """
    Affiche un QCM interactif avec ipywidgets.
    correct : index (int) ou liste d'index de la/des bonne(s) réponse(s).
    """
    if isinstance(correct, int):
        correct = [correct]

    boutons = w.RadioButtons(options=options, layout=w.Layout(width='100%'))
    sortie  = w.Output()
    titre   = w.HTML(f"<b>❓ {question}</b>")

    def verifier(change):
        idx = options.index(change['new'])
        sortie.clear_output()
        with sortie:
            if idx in correct:
                display(w.HTML(
                    f"<div style='padding:10px;border-radius:8px;background:#d4edda;color:#155724;border-left:5px solid #28a745'>"
                    f"✅ <b>Bravo, c'est exact !</b><br>{explication}</div>"
                ))
            else:
                humour = _random_qcm.choice(_RELANCES_HUMOUR)
                display(w.HTML(
                    f"<div style='padding:10px;border-radius:8px;background:#f8d7da;color:#721c24;border-left:5px solid #dc3545'>"
                    f"❌ <b>Pas tout à fait...</b> {humour}<br><br>"
                    f"<b>Explication :</b> {explication}</div>"
                ))

    boutons.observe(verifier, names='value')
    display(w.VBox([titre, boutons, sortie]))

def check(nom, valeur, attendu, tol=1e-5):
    """
    Vérifie la valeur d'une variable ou d'un array calculé dans un exercice.
    Affiche un retour visuel vert ou rouge.
    """
    ok = False
    try:
        if isinstance(attendu, np.ndarray) and isinstance(valeur, np.ndarray):
            ok = valeur.shape == attendu.shape and np.allclose(valeur, attendu, atol=tol)
        elif isinstance(attendu, (int, float, np.integer, np.floating)) and isinstance(valeur, (int, float, np.integer, np.floating)):
            ok = abs(valeur - attendu) <= tol
        elif isinstance(attendu, (tuple, list)) and isinstance(valeur, (tuple, list)):
            ok = len(valeur) == len(attendu) and all(abs(v - a) <= tol for v, a in zip(valeur, attendu))
        else:
            ok = bool(valeur == attendu)
    except Exception:
        ok = False

    if ok:
        display(w.HTML(
            f"<div style='padding:8px 12px;border-radius:6px;background:#d4edda;color:#155724;border-left:4px solid #28a745'>"
            f"✅ <b>{nom}</b> est correct ! (Valeur : <code>{repr(valeur)}</code>)</div>"
        ))
    else:
        display(w.HTML(
            f"<div style='padding:8px 12px;border-radius:6px;background:#f8d7da;color:#721c24;border-left:4px solid #dc3545'>"
            f"❌ <b>{nom}</b> n'est pas encore la valeur attendue.<br>"
            f"Ta valeur : <code>{repr(valeur)}</code><br>"
            f"Valeur attendue : <code>{repr(attendu)}</code></div>"
        ))

print(f"✅ Environnement initialisé avec NumPy version {np.__version__}")
print("✅ Fonctions qcm() et check() prêtes à l'emploi.")


✅ Environnement initialisé avec NumPy version 2.5.1
✅ Fonctions qcm() et check() prêtes à l'emploi.


---
## 1 · Du Neurone Biologique au Neurone Formel

En 1943, le neurophysiologiste **Warren McCulloch** et le logicien **Walter Pitts** publient une première modélisation simplifiée du fonctionnement d'un neurone cérébral. En 1957, **Frank Rosenblatt** y ajoute une règle d'apprentissage automatique : le **Perceptron** est né.

### 🧠 L'analogie biologique

![Analogie Biologique](pictures/perceptron_bio_analogy.png)

### ⚙️ L'Architecture & le Flux de Calcul

![Architecture du Perceptron](pictures/perceptron_architecture.png)

### 📐 Les Équations Fondamentales

Pour un échantillon d'entrée représenté par un vecteur de $m$ caractéristiques $\mathbf{x} = [x_1, x_2, \dots, x_m]$ :

1. **La combinaison linéaire (somme pondérée + biais)** :
   $$z = \sum_{j=1}^m w_j x_j + b = \mathbf{w} \cdot \mathbf{x} + b$$
   - $\mathbf{w}$ est le vecteur des **poids** (*weights*). Il mesure l'importance relative de chaque caractéristique.
   - $b$ est le **biais** (*bias*). C'est un seuil d'activation intrinsèque.

2. **La fonction d'activation (Échelon de Heaviside)** :
   $$\hat{y} = f(z) = \begin{cases} 1 & \text{si } z \ge 0 \\ 0 & \text{si } z < 0 \end{cases}$$

### ⚡ La Formulation Matricielle NumPy (Broadcasting & Produit Matriciel)

Si l'on dispose d'un jeu de données de **$N$ observations** et **$M$ caractéristiques** stocké dans une matrice $\mathbf{X}$ de forme $(N, M)$ :
$$\mathbf{Z} = \mathbf{X} \mathbf{w} + b$$
- $\mathbf{X}$ a pour forme `(N, M)`
- $\mathbf{w}$ a pour forme `(M,)`
- En NumPy, le produit matriciel `X @ w` donne un vecteur de forme `(N,)`.
- L'addition du scalaire `b` s'effectue automatiquement par **broadcasting** sur les $N$ lignes !
- La fonction d'activation s'applique instantanément avec `np.where(z >= 0, 1, 0)`.


In [5]:
# Programme 2 — QCM 1 : Rôle du Biais
qcm(
    "À quoi sert le paramètre de biais b dans la combinaison linéaire z = w · x + b ?",
    [
        "À normaliser les données d'entrée pour qu'elles restent comprises entre 0 et 1.",
        "À décaler la frontière de décision pour qu'elle ne passe pas obligatoirement par l'origine (0, 0).",
        "À garantir que les poids synaptiques w restent toujours strictement positifs.",
        "À accélérer le calcul vectorisé du produit matriciel en mémoire."
    ],
    correct=1,
    explication="Sans biais (b = 0), la condition w · x = 0 impose que la frontière passe obligatoirement par (0, 0). Le biais b permet de translater la droite de décision n'importe où dans le plan !"
)


### 💡 Exemple Concret — La Formulation Matricielle en Action

#### 🔌 D'où viennent les observations et les caractéristiques ?
Dans le monde réel (comme sur les séries temporelles du réseau électrique ou d'un compteur Linky) :
1. Le capteur mesure un flux continu de puissance dans le temps (un point toutes les 10 secondes).
2. Pour analyser ce signal en Machine Learning, on le découpe en **fenêtres de temps** (par exemple des créneaux de 5 minutes). **Chaque fenêtre devient UNE observation (une ligne de la matrice $\mathbf{X}$)**.
3. Sur chaque fenêtre, on résume la forme du signal par des indicateurs statistiques (les **colonnes / features**) :
   - **Feature 1 ($x_1$)** : la puissance moyenne mesurée sur la fenêtre (en Watts).
   - **Feature 2 ($x_2$)** : la puissance maximale (le pic) observée sur la fenêtre (en Watts).

> 📌 **Question posée au neurone** : *« Durant ce créneau de 5 minutes, l'appareil était-il allumé ($\hat{y} = 1$) ou éteint / en veille ($\hat{y} = 0$) ? »*

Voyons comment NumPy évalue **plusieurs fenêtres d'un seul coup** grâce au produit matriciel `X @ w` et au **broadcasting** du biais `+ b`, sans aucune boucle `for` :


In [6]:
# Programme 3 — Exemple Concret : Formulation Matricielle (X @ w + b)

# 1. Matrice X de forme (N=3, M=2) : 3 fenêtres temporelles, 2 features par fenêtre
# Colonne 0 = Puissance moyenne (W) | Colonne 1 = Puissance maximale / Pic (W)
X_exemple = np.array([
    [120.0, 150.0],  # Fenêtre 1 (14h00-14h05) : Forte consommation (ex: compresseur ou four en marche)
    [ 10.0,  15.0],  # Fenêtre 2 (14h05-14h10) : Consommation quasi-nulle (veille résiduelle)
    [ 70.0,  90.0]   # Fenêtre 3 (14h10-14h15) : Consommation modérée (appareil actif)
])

# 2. Vecteur des poids w (importance de chaque variable) et Biais b (seuil d'activation)
w_exemple = np.array([0.02, 0.01])  # Poids associés à [moyenne, pic]
b_exemple = -2.0                    # Biais / seuil de décision

# 3. Étape A : Produit matriciel vectorisé X @ w
# (3, 2) @ (2,) -> vecteur de forme (3,)
z_produit = X_exemple @ w_exemple

# 4. Étape B : Addition du biais (+ b) par BROADCASTING
# Le scalaire b_exemple (-2.0) est automatiquement ajouté aux 3 fenêtres
z_exemple = z_produit + b_exemple

# 5. Étape C : Fonction de seuil de Heaviside (1 si z >= 0, sinon 0)
y_pred_exemple = np.where(z_exemple >= 0, 1, 0)

# 6. Affichage pédagogique détaillé des résultats
print("===============================================================")
print("📊 FORMULATION MATRICIELLE NUMPY (X @ w + b)")
print("===============================================================")
print(f"Dimensions de X : {X_exemple.shape}  (3 fenêtres de 5 min, 2 features)")
print(f"Dimensions de w : {w_exemple.shape}       (2 poids synaptiques)")
print(f"Biais b         : {b_exemple}       (scalaire seuil)")
print("---------------------------------------------------------------")
print(f"1. Produit matriciel X @ w    : {z_produit}")
print(f"2. Somme totale z = X @ w + b : {z_exemple}  <-- (+ b par broadcasting)")
print(f"3. Prédiction y_hat (z >= 0)  : {y_pred_exemple}        (1=Actif/ON, 0=Inactif/OFF)")
print("---------------------------------------------------------------")

labels_fenetres = ["14h00-14h05", "14h05-14h10", "14h10-14h15"]
for i, (fen, x_i, z_i, pred_i) in enumerate(zip(labels_fenetres, X_exemple, z_exemple, y_pred_exemple), start=1):
    etat = "✅ Appareil ON (1)" if pred_i == 1 else "❌ Appareil OFF (0)"
    print(f"  • Fenêtre {i} ({fen}) : [Moy={x_i[0]}W, Pic={x_i[1]}W] ➔ z={z_i:+.2f} ➔ {etat}")

print("===============================================================")
print(f"🎯 RÉSULTAT FINAL : Prédictions y_hat = {y_pred_exemple}")
print("===============================================================")


📊 FORMULATION MATRICIELLE NUMPY (X @ w + b)
Dimensions de X : (3, 2)  (3 fenêtres de 5 min, 2 features)
Dimensions de w : (2,)       (2 poids synaptiques)
Biais b         : -2.0       (scalaire seuil)
---------------------------------------------------------------
1. Produit matriciel X @ w    : [3.9  0.35 2.3 ]
2. Somme totale z = X @ w + b : [ 1.9  -1.65  0.3 ]  <-- (+ b par broadcasting)
3. Prédiction y_hat (z >= 0)  : [1 0 1]        (1=Actif/ON, 0=Inactif/OFF)
---------------------------------------------------------------
  • Fenêtre 1 (14h00-14h05) : [Moy=120.0W, Pic=150.0W] ➔ z=+1.90 ➔ ✅ Appareil ON (1)
  • Fenêtre 2 (14h05-14h10) : [Moy=10.0W, Pic=15.0W] ➔ z=-1.65 ➔ ❌ Appareil OFF (0)
  • Fenêtre 3 (14h10-14h15) : [Moy=70.0W, Pic=90.0W] ➔ z=+0.30 ➔ ✅ Appareil ON (1)
🎯 RÉSULTAT FINAL : Prédictions y_hat = [1 0 1]


### 🧐 À quoi sert ce programme et que signifient les prédictions $\hat{y}$ ?

Ce petit programme illustre la **phase d'inférence (ou propagation avant / *forward pass*)** d'un neurone artificiel.

#### 1. Le but concret du programme
Dans la pratique, un modèle de Machine Learning sert à **automatiser une décision binaire** à partir de signaux ou caractéristiques brutes :
- **La question posée** : *« Au vu des mesures électriques constatées (puissance moyenne et pic), l'appareil est-il en état de marche ($1$) ou éteint / en veille ($0$) ? »*
- **La puissance de NumPy** : Plutôt que de tester chaque observation dans une boucle `for` lente, la formule `np.where(X @ w + b >= 0, 1, 0)` calcule la décision pour **des millions d'échantillons en une seule opération vectorisée**.

#### 2. Décorticage du vecteur de prédiction $\mathbf{\hat{y}}$

Le modèle a produit le vecteur binaire :  
$$\mathbf{\hat{y}} = [1, 0, 1]$$

Voici le détail du calcul pour chaque observation :

| Observation | Caractéristiques $\mathbf{x}_i$ | Somme $z_i = \mathbf{w} \cdot \mathbf{x}_i + b$ | Condition $z_i \ge 0$ | Prédiction $\hat{y}_i$ | Signification Métier |
|:---:|:---:|:---:|:---:|:---:|:---|
| **Obs 1** | $[120\text{ W}, 150\text{ W}]$ | $120 \times 0.02 + 150 \times 0.01 - 2.0 = \mathbf{+1.90}$ | Vraie ($\ge 0$) | **$\hat{y}_1 = 1$** | ✅ Appareil **Actif / ON** |
| **Obs 2** | $[10\text{ W}, 15\text{ W}]$ | $10 \times 0.02 + 15 \times 0.01 - 2.0 = \mathbf{-1.65}$ | Fausse ($< 0$) | **$\hat{y}_2 = 0$** | ❌ Appareil **Inactif / Veille** |
| **Obs 3** | $[70\text{ W}, 90\text{ W}]$ | $70 \times 0.02 + 90 \times 0.01 - 2.0 = \mathbf{+0.30}$ | Vraie ($\ge 0$) | **$\hat{y}_3 = 1$** | ✅ Appareil **Actif / ON** |

#### 3. La passerelle vers la suite : Pourquoi $\hat{y}$ est au cœur de l'apprentissage ?
Dans cet exemple, les poids $\mathbf{w}$ et le biais $b$ étaient fixés à l'avance. Mais comment le Perceptron peut-il **trouver tout seul** les bons poids sans qu'un humain ne les définisse ?

C'est précisément l'objet de la section suivante : comparer cette prédiction $\hat{y}_i$ avec la réalité attendue $y_i$ pour calculer l'**erreur $e_i = y_i - \hat{y}_i$** et corriger les poids automatique lorsque le neurone se trompe !


---
## 2 · La Règle d'Apprentissage de Rosenblatt (1957)

Un neurone sans apprentissage n'est qu'une formule statique. La grande percée de Rosenblatt a été de proposer un algorithme permettant au Perceptron d'**ajuster ses poids automatiquement** à partir de ses erreurs.

### 🔄 L'Algorithme d'Apprentissage pas à pas

On présente au neurone chaque exemple $(\mathbf{x}_i, y_i)$ de notre jeu de données, où $y_i \in \{0, 1\}$ est la classe réelle attendue.

1. **Prédiction** : le modèle calcule sa prédiction actuelle $\hat{y}_i = f(\mathbf{w} \cdot \mathbf{x}_i + b)$.
2. **Calcul de l'erreur** :
   $$e_i = y_i - \hat{y}_i$$
   Comme $y_i$ et $\hat{y}_i$ valent soit $0$, soit $1$, l'erreur $e_i$ ne peut prendre que **3 valeurs possibles** :
   - $e_i = 0$ : Prédiction correcte.
   - $e_i = +1$ : Le modèle a prédit $0$ alors qu'il fallait $1$ (**Faux Négatif**).
   - $e_i = -1$ : Le modèle a prédit $1$ alors qu'il fallait $0$ (**Faux Positif**).
3. **Mise à jour des poids et du biais** :
   $$\mathbf{w} \leftarrow \mathbf{w} + \eta \cdot (y_i - \hat{y}_i) \cdot \mathbf{x}_i$$
   $$b \leftarrow b + \eta \cdot (y_i - \hat{y}_i)$$
   où $\eta$ (*learning rate*, taux d'apprentissage) est un petit nombre positif (ex: $0.1$ ou $0.01$).

### 🧭 L'Intuition Physique derrière la Règle

| $y_i$ (Réel) | $\hat{y}_i$ (Prédit) | Erreur $e_i$ | Action sur les poids et le biais | Pourquoi cette correction ? |
|---|---|---|---|---|
| **1** | **1** | **0** | Aucune modification | Tout va bien, le neurone a vu juste. |
| **0** | **0** | **0** | Aucune modification | Tout va bien. |
| **1** | **0** | **+1** | $\mathbf{w} \leftarrow \mathbf{w} + \eta \mathbf{x}_i$ <br> $b \leftarrow b + \eta$ | $z$ était trop petit ($<0$). En ajoutant $\eta \mathbf{x}_i$, on **augmente** la valeur future de $\mathbf{w} \cdot \mathbf{x}_i$. |
| **0** | **1** | **-1** | $\mathbf{w} \leftarrow \mathbf{w} - \eta \mathbf{x}_i$ <br> $b \leftarrow b - \eta$ | $z$ était trop grand ($\ge 0$). En soustrayant $\eta \mathbf{x}_i$, on **diminue** la valeur future de $\mathbf{w} \cdot \mathbf{x}_i$. |

> 📜 **Théorème de convergence du Perceptron (Novikoff, 1962)** :  
> Si les données sont **linéairement séparables**, le Perceptron est mathématiquement garanti de trouver une solution exacte qui sépare parfaitement les deux classes en un **nombre fini d'étapes** !


In [4]:
# Programme 4 — QCM 2 : Mise à jour des poids
qcm(
    "Un exemple d'entrée a pour caractéristiques x = [2.0, 3.0] et pour label réel y = 1. Le Perceptron calcule z = -0.4 et prédit donc y_hat = 0. Avec un learning rate de 0.1, quelle est la modification apportée aux poids w ?",
    [
        "On soustrait [-0.2, -0.3] à w car le modèle a surestimé la classe.",
        "On ajoute [+0.2, +0.3] à w et on ajoute 0.1 au biais b.",
        "Les poids ne changent pas car l'erreur z = -0.4 est trop proche de 0.",
        "On multiplie w par 0.1."
    ],
    correct=1,
    explication="L'erreur vaut (y - y_hat) = 1 - 0 = +1. La mise à jour est delta_w = lr * 1 * x = 0.1 * [2.0, 3.0] = [+0.2, +0.3]. Le biais augmente aussi de 0.1 * 1 = +0.1."
)


---
## 3 · Géométrie du Perceptron : La Droite de Séparation

Pour visualiser ce que fait le Perceptron, plaçons-nous dans le cas intuitif où chaque observation possède **2 caractéristiques** : $\mathbf{x} = [x_1, x_2]$.

Le neurone calcule :
$$z = w_1 x_1 + w_2 x_2 + b$$

La frontière entre la région où il prédit $1$ ($z \ge 0$) et la région où il prédit $0$ ($z < 0$) est la droite critique d'équation **$z = 0$** :

$$w_1 x_1 + w_2 x_2 + b = 0$$

En isolant $x_2$ en fonction de $x_1$ (sous la forme classique $y = m x + p$) :

$$x_2 = -\frac{w_1}{w_2} x_1 - \frac{b}{w_2}$$

![Géométrie du Perceptron 2D](pictures/perceptron_geometrie_2d.png)

- La **pente** de la droite de séparation est $-\frac{w_1}{w_2}$.
- L'**ordonnée à l'origine** est $-\frac{b}{w_2}$.
- Le vecteur des poids $\mathbf{w} = [w_1, w_2]$ est le **vecteur orthogonal (normal)** à la droite, orienté vers la zone des prédictions positives ($+1$).

Lors de l'apprentissage, chaque mise à jour fait pivoter et translater cette droite jusqu'à ce que tous les points bleus soient d'un côté et tous les points rouges de l'autre !

### 🎬 En direct : L'Évolution Dynamique de la Droite pendant l'Apprentissage

Voici concrètement comment la droite de séparation pivote (orientation du vecteur normal $\mathbf{w}$) et translate (ajustement du biais $b$) époque après époque pour séparer deux nuages de points 2D :

<p align="center">
  <img src="pictures/perceptron_apprentissage.gif" alt="Animation de l'apprentissage du Perceptron" width="92%" style="border-radius: 8px; box-shadow: 0 4px 16px rgba(0,0,0,0.3);" />
</p>

> [!TIP]
> **Ce que montre l'animation synchronisée :**
> - **À gauche (Espace 2D)** : La droite démarre arbitrairement près de $(0,0)$ avec un biais nul ($b=0$). À chaque échantillon mal classé (marqué par un anneau rouge), le vecteur $\mathbf{w}$ tourne et le biais $b$ translate la droite vers le haut jusqu'à trouver un couloir séparateur optimal avec **0 erreur**.
> - **À droite (Courbe d'apprentissage)** : Le nombre d'erreurs d'échantillons chute progressivement au fil des 27 époques jusqu'à atteindre une séparation parfaite (**100 % de précision**).
>
> *(Une version vidéo Full HD est disponible dans [`pictures/perceptron_apprentissage.mp4`](pictures/perceptron_apprentissage.mp4)).*


---
## 4 · À Toi de Jouer : Coder le Perceptron en pur NumPy

Nous allons maintenant construire notre Perceptron brique par brique à l'aide de NumPy :
1. **Exercice 1** : la propagation avant vectorisée (`forward pass`).
2. **Exercice 2** : la fonction de correction d'un échantillon (`update_step`).
3. **Exercice 3** : la classe complète `PerceptronNumPy` avec ses méthodes `.fit()` et `.predict()`.


### ✏️ Exercice 1 — La Propagation Avant Vectorisée (`forward`)

Dans cet exercice, nous disposons d'une matrice `X_demo` de 4 observations et 2 variables, d'un vecteur de poids `w_demo` et d'un biais `b_demo`.

1. Calculez la combinaison linéaire `z_demo` pour toutes les observations d'un coup grâce à l'opérateur matriciel `@` de NumPy :
   $$\mathbf{Z} = \mathbf{X} \mathbf{w} + b$$
2. Calculez le vecteur des prédictions binaires `y_pred_demo` en utilisant `np.where(condition, 1, 0)` :
   - Vaut `1` si $z \ge 0$
   - Vaut `0` si $z < 0$


In [ ]:
# Programme 5 — Exercice 1 : Propagation avant vectorisée
X_demo = np.array([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0]
])
w_demo = np.array([0.6, 0.6])
b_demo = -0.8

# Remplacez les '...' par votre code :
z_demo = ...
y_pred_demo = ...


SyntaxError: invalid syntax (1983509134.py, line 13)

In [ ]:
# Programme 6 — Test Exercice 1
_z_ref = X_demo @ w_demo + b_demo
_y_ref = np.where(_z_ref >= 0, 1, 0)

check("z_demo", z_demo, _z_ref)
check("y_pred_demo", y_pred_demo, _y_ref)


### ✏️ Exercice 2 — Le Pas d'Ajustement d'un Échantillon (`update_step`)

Écrivons la fonction responsable d'ajuster les poids pour une observation unique `(x_i, y_i)` :

1. Calculez la prédiction scalaire `pred` : elle vaut `1` si $\mathbf{w} \cdot \mathbf{x}_i + b \ge 0$, et `0` sinon (utilisez `np.dot` ou le produit scalaire).
2. Calculez l'erreur `delta = y_i - pred`.
3. Si `delta != 0`, appliquez la règle de Rosenblatt :
   - `w_new = w + lr * delta * x_i`
   - `b_new = b + lr * delta`
4. Retournez `(w_new, b_new, 1)` si une erreur a eu lieu, ou `(w, b, 0)` si la prédiction était déjà exacte.


In [ ]:
# Programme 7 — Exercice 2 : Règle de mise à jour unitaire
def update_step(w, b, x_i, y_i, lr=0.1):
    """
    Effectue une étape de mise à jour pour un échantillon (x_i, y_i).
    Retourne : (w_mis_a_jour, b_mis_a_jour, erreur_commise)
    """
    # Remplacez les '...' par votre code :
    z = np.dot(x_i, w) + b
    pred = 1 if z >= 0 else 0
    delta = y_i - pred
    
    if delta != 0:
        w_new = ...
        b_new = ...
        erreur = 1
    else:
        w_new = w
        b_new = b
        erreur = 0
        
    return w_new, b_new, erreur


In [ ]:
# Programme 8 — Test Exercice 2
# Cas 1 : Le modèle commet une erreur (y=1 alors que z=-0.5 < 0)
_w_initial = np.array([0.0, 0.0])
_b_initial = -0.5
_xi = np.array([2.0, 3.0])
_yi = 1

_w_corrige, _b_corrige, _err1 = update_step(_w_initial.copy(), _b_initial, _xi, _yi, lr=0.1)

check("w après correction", _w_corrige, np.array([0.2, 0.3]))
check("b après correction", _b_corrige, -0.4)
check("erreur détectée (doit valoir 1)", _err1, 1)

# Cas 2 : Le modèle a vu juste (pas de modification)
_w_ok, _b_ok, _err0 = update_step(np.array([0.5, 0.5]), 0.2, np.array([1.0, 1.0]), 1, lr=0.1)
check("w inchangé si correct", _w_ok, np.array([0.5, 0.5]))
check("erreur nulle si correct (doit valoir 0)", _err0, 0)


### ✏️ Exercice 3 — La Classe Complète `PerceptronNumPy`

Assemblons à présent ces briques dans une classe Python moderne respectant les conventions de Scikit-Learn :
- `__init__(self, lr=0.1, n_epochs=20, random_state=42)` : initialise les hyperparamètres.
- `predict(self, X)` : calcule la prédiction vectorisée pour une matrice `X` de taille `(N, M)`.
- `fit(self, X, y)` :
  1. Initialise les poids `self.w` avec de petites valeurs aléatoires normales via `rng = np.random.default_rng(self.random_state)` et `self.b = 0.0`.
  2. Pour chaque époque dans `range(self.n_epochs)` :
     - Parcourt chaque exemple `(xi, target)` dans `zip(X, y)`.
     - Calcule la prédiction scalaire pour `xi` et l'écart `delta = target - pred`.
     - Si `delta != 0`, met à jour `self.w` et `self.b`, et incrémente le compteur `nb_erreurs`.
     - Ajoute `nb_erreurs` à la liste d'historique `self.errors_`.
     - **Arrêt anticipé (Early Stopping)** : si `nb_erreurs == 0`, toutes les observations sont correctement classées ! On quitte la boucle d'apprentissage avec `break`.


In [ ]:
# Programme 9 — Exercice 3 : Classe PerceptronNumPy
class PerceptronNumPy:
    def __init__(self, lr=0.1, n_epochs=20, random_state=42):
        self.lr = lr
        self.n_epochs = n_epochs
        self.random_state = random_state
        self.w = None
        self.b = 0.0
        self.errors_ = []

    def predict(self, X):
        """
        Calcule les prédictions binaires {0, 1} pour une matrice X de forme (N, M).
        """
        # Remplacez les '...' par votre code vectorisé avec @ et np.where :
        z = ...
        return ...

    def fit(self, X, y):
        """
        Entraîne le Perceptron sur les données d'entraînement X et les labels y.
        """
        rng = np.random.default_rng(self.random_state)
        # Initialisation aléatoire des poids pour M caractéristiques :
        self.w = rng.normal(loc=0.0, scale=0.01, size=X.shape[1])
        self.b = 0.0
        self.errors_ = []

        for epoch in range(self.n_epochs):
            nb_erreurs = 0
            for xi, target in zip(X, y):
                # Calcul de la prédiction scalaire pour l'échantillon xi :
                z_i = np.dot(xi, self.w) + self.b
                pred = 1 if z_i >= 0 else 0
                delta = target - pred

                # Mise à jour si erreur :
                if delta != 0:
                    self.w += ...
                    self.b += ...
                    nb_erreurs += 1

            self.errors_.append(nb_erreurs)
            
            # Arrêt anticipé si convergence parfaite :
            if nb_erreurs == 0:
                break

        return self


In [ ]:
# Programme 10 — Test Exercice 3
# Testons la classe sur un petit jeu de test
_X_test = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
_y_test = np.array([0, 0, 0, 1])  # Porte AND

_p = PerceptronNumPy(lr=0.1, n_epochs=20, random_state=42)
_p.fit(_X_test, _y_test)
_preds = _p.predict(_X_test)

check("Perceptron converge sur AND", np.array_equal(_preds, _y_test), True)
check("Historique des erreurs enregistré", len(_p.errors_) > 0, True)
check("Erreurs finales nulles", _p.errors_[-1], 0)


---
## 5 · Application 1 — Les Portes Logiques (AND, OR)

Les portes logiques ont été les premiers bancs d'essai de Frank Rosenblatt dans les années 1950.
Une porte logique prend deux entrées binaires $x_1, x_2 \in \{0, 1\}$ et produit une sortie binaire $y \in \{0, 1\}$.

| $x_1$ | $x_2$ | ET logique (AND) | OU logique (OR) | OU exclusif (XOR) |
|:---:|:---:|:---:|:---:|:---:|
| 0 | 0 | **0** | **0** | **0** |
| 0 | 1 | **0** | **1** | **1** |
| 1 | 0 | **0** | **1** | **1** |
| 1 | 1 | **1** | **1** | **0** |

Voyons comment notre `PerceptronNumPy` résout le AND et le OR !


In [ ]:
# Programme 11 — Entraînement sur les Portes AND et OR
X_logic = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

y_and = np.array([0, 0, 0, 1])
y_or  = np.array([0, 1, 1, 1])

# 1. Entraînement sur la porte AND
p_and = PerceptronNumPy(lr=0.1, n_epochs=20, random_state=42)
p_and.fit(X_logic, y_and)

# 2. Entraînement sur la porte OR
p_or = PerceptronNumPy(lr=0.1, n_epochs=20, random_state=42)
p_or.fit(X_logic, y_or)

print("🎯 Résultats Porte AND :")
print("  Prédictions :", p_and.predict(X_logic), " Attendu :", y_and)
print(f"  Poids w : {p_and.w.round(3)}, Biais b : {round(p_and.b, 3)}")
print(f"  Convergence en {len(p_and.errors_)} époques.")

print("\n🎯 Résultats Porte OR :")
print("  Prédictions :", p_or.predict(X_logic), " Attendu :", y_or)
print(f"  Poids w : {p_or.w.round(3)}, Biais b : {round(p_or.b, 3)}")
print(f"  Convergence en {len(p_or.errors_)} époques.")


In [ ]:
# Programme 12 — Visualisation Graphique des Portes AND et OR
def tracer_decision_2d(ax, model, X, y, titre):
    """
    Trace les points de données, les régions de décision et la droite frontière.
    """
    # 1. Grille fine pour colorer les régions de décision
    x_min, x_max = -0.5, 1.5
    y_min, y_max = -0.5, 1.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.25, cmap='coolwarm')

    # 2. Droite de décision théorique : w1*x1 + w2*x2 + b = 0 => x2 = -(w1*x1 + b) / w2
    if abs(model.w[1]) > 1e-5:
        x1_line = np.array([x_min, x_max])
        x2_line = -(model.w[0] * x1_line + model.b) / model.w[1]
        ax.plot(x1_line, x2_line, 'r--', lw=2.5, label='Frontière z = 0')

    # 3. Tracé des points
    ax.scatter(X[y == 0, 0], X[y == 0, 1], color='#0969da', s=120, edgecolors='black', lw=1.5, label='Classe 0', zorder=5)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], color='#cf222e', s=140, marker='^', edgecolors='black', lw=1.5, label='Classe 1', zorder=5)

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel('$x_1$', fontsize=12)
    ax.set_ylabel('$x_2$', fontsize=12)
    ax.set_title(titre, fontsize=13, fontweight='bold')
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='upper left')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
tracer_decision_2d(ax1, p_and, X_logic, y_and, "Porte ET (AND) — Séparation Linéaire")
tracer_decision_2d(ax2, p_or, X_logic, y_or, "Porte OU (OR) — Séparation Linéaire")
plt.tight_layout()
plt.show()


---
## 6 · Le Drame du XOR et le Premier Hiver de l'IA (1969)

Après l'enthousiasme suscité par les succès du Perceptron, un événement majeur va brutalement freiner la recherche en intelligence artificielle.

En 1969, deux pionniers du MIT, **Marvin Minsky** et **Seymour Papert**, publient le livre de référence *Perceptrons*. Ils y démontrent une limitation mathématique rédhibitoire : **un Perceptron à couche unique est incapable d'apprendre la fonction OU Exclusif (XOR)** !

### 💥 Testons le XOR avec notre Perceptron


In [ ]:
# Programme 13 — Tentative d'entraînement sur la Porte XOR
y_xor = np.array([0, 1, 1, 0])

p_xor = PerceptronNumPy(lr=0.1, n_epochs=30, random_state=42)
p_xor.fit(X_logic, y_xor)

print("🎯 Résultats Porte XOR :")
print("  Prédictions :", p_xor.predict(X_logic))
print("  Attendu     :", y_xor)
print("  Historique des erreurs sur 30 époques :", p_xor.errors_)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Graphique 1 : Évolution des erreurs
ax1.plot(range(1, len(p_xor.errors_) + 1), p_xor.errors_, marker='o', color='#cf222e', lw=2)
ax1.set_title("XOR : Erreurs par Époque (Oscillations perpétuelles)", fontweight='bold')
ax1.set_xlabel("Époque")
ax1.set_ylabel("Nombre d'erreurs (sur 4 exemples)")
ax1.set_ylim(0, 5)
ax1.grid(True, linestyle=':', alpha=0.6)

# Graphique 2 : Géométrie impossible du XOR
tracer_decision_2d(ax2, p_xor, X_logic, y_xor, "XOR : Aucune droite ne peut séparer les classes !")
plt.tight_layout()
plt.show()


In [ ]:
# Programme 14 — QCM 3 : La Séparabilité Linéaire
qcm(
    "Pourquoi le Perceptron simple de Rosenblatt est-il mathématiquement incapable d'apprendre la fonction XOR ?",
    [
        "Parce que le learning rate par défaut (0.1) est trop élevé et fait diverger le calcul.",
        "Parce qu'il n'existe aucune droite (hyperplan séparateur) capable d'isoler les points (0,1) et (1,0) des points (0,0) et (1,1).",
        "Parce que le jeu de données ne compte pas assez d'échantillons pour converger.",
        "Parce que la fonction de seuil de Heaviside ne s'applique pas aux nombres négatifs."
    ],
    correct=1,
    explication="Le problème du XOR n'est PAS linéairement séparable : les points d'une même classe sont situés sur des diagonales opposées. Pour résoudre le XOR, il faut combiner au moins DEUX neurones (un réseau multicouches ou MLP) !"
)


---
## 7 · Grand Cas Pratique — Classification 2D de Données Synthétiques (Blobs NumPy)

Sortons des 4 exemples simplistes des portes logiques et testons notre `PerceptronNumPy` sur un véritable problème de classification binaire à grande échelle : **100 observations 2D** réparties en deux nuages gaussiens (*blobs*).

Nous allons générer ces données nous-mêmes grâce à l'API aléatoire moderne de NumPy (`np.random.default_rng`), vue dans **Tool 2** !

- **Classe 0** : 50 points distribués autour du centre $(2.0, 2.0)$ avec un écart-type de $0.6$.
- **Classe 1** : 50 points distribués autour du centre $(5.0, 5.0)$ avec un écart-type de $0.6$.


In [ ]:
# Programme 15 — Génération du Dataset Synthétique 2D (Blobs)
rng_data = np.random.default_rng(seed=42)

# Génération des 2 nuages de points (50 échantillons par classe)
X_classe_0 = rng_data.normal(loc=[2.0, 2.0], scale=0.6, size=(50, 2))
X_classe_1 = rng_data.normal(loc=[5.0, 5.0], scale=0.6, size=(50, 2))

# Empilement vertical en une seule matrice X de forme (100, 2)
X_blobs = np.vstack([X_classe_0, X_classe_1])

# Vecteur des labels y (50 zéros suivis de 50 uns)
y_blobs = np.array([0] * 50 + [1] * 50)

print("Forme de la matrice X_blobs :", X_blobs.shape)
print("Forme du vecteur y_blobs    :", y_blobs.shape)
print(f"Nombre d'échantillons Classe 0 : {np.sum(y_blobs == 0)}, Classe 1 : {np.sum(y_blobs == 1)}")


### ✏️ Exercice 4 — Entraîner et Évaluer le Perceptron sur les Blobs

1. Instanciez un objet `p_blobs` de la classe `PerceptronNumPy` avec :
   - `lr=0.05`
   - `n_epochs=50`
   - `random_state=42`
2. Entraînez le modèle sur `X_blobs` et `y_blobs` avec la méthode `.fit()`.
3. Calculez les prédictions du modèle sur l'ensemble des données dans `y_pred_blobs`.
4. Calculez le taux de précision global `accuracy_blobs` :
   $$\text{Accuracy} = \frac{\text{Nombre de prédictions correctes}}{\text{Nombre total d'échantillons}} = \text{np.mean}(y_{pred} == y)$$


In [ ]:
# Programme 16 — Exercice 4 : Entraînement sur les Blobs 2D
# Remplacez les '...' par votre code :
p_blobs = ...

# Entraînement :
...

# Prédiction et Accuracy :
y_pred_blobs = ...
accuracy_blobs = ...

print(f"🎯 Précision obtenue : {accuracy_blobs * 100:.1f} %")
print(f"⚡ Nombre d'époques pour converger : {len(p_blobs.errors_)}")


In [ ]:
# Programme 17 — Test Exercice 4
check("accuracy_blobs", accuracy_blobs, 1.0)
check("Modèle p_blobs bien entraîné", hasattr(p_blobs, 'w') and p_blobs.w is not None, True)
check("Convergence parfaite atteinte (dernière époque à 0 erreur)", p_blobs.errors_[-1] if hasattr(p_blobs, 'errors_') and p_blobs.errors_ else -1, 0)


In [ ]:
# Programme 18 — Visualisation Spectaculaire de la Frontière et de la Convergence
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# 1. Frontière de décision et zones colorées
tracer_decision_2d(ax1, p_blobs, X_blobs, y_blobs, "Frontière de Décision Apprise par le Perceptron")

# 2. Courbe de convergence (Historique des erreurs)
ax2.plot(range(1, len(p_blobs.errors_) + 1), p_blobs.errors_, marker='s', color='#1a7f37', lw=2, label="Erreurs par époque")
ax2.set_title("Courbe d'Apprentissage : Convergence vers 0 Erreur", fontsize=13, fontweight='bold')
ax2.set_xlabel("Époque", fontsize=12)
ax2.set_ylabel("Nombre d'erreurs d'échantillons", fontsize=12)
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()


### 🎥 Animation Dynamique — Évolution de la Frontière pendant l'Apprentissage

Voici l'animation complète montrant l'évolution pas à pas de la frontière de décision $w_1 x_1 + w_2 x_2 + b = 0$ et de la courbe d'apprentissage sur nos 100 observations pendant les 27 époques :

<p align="center">
  <img src="pictures/perceptron_apprentissage.gif" alt="Animation apprentissage Perceptron" width="95%" style="border-radius: 8px; box-shadow: 0 4px 16px rgba(0,0,0,0.3);" />
</p>

> [!TIP]
> **Que constatez-vous sur l'animation ?**
> 1. **Initialisation (Époque 0)** : Les poids démarrent très proches de zéro ($w \approx [0, 0]$, $b = 0$), la droite sépare arbitrairement l'espace et 50 points sont mal classés.
> 2. **Rotation et Translation (Époques 1 à 26)** : À chaque échantillon mal classé, le vecteur normal $\mathbf{w}$ pivote ($\Delta \mathbf{w} = \eta (y - \hat{y}) \mathbf{x}$) et le biais $b$ translate la droite vers le haut, éliminant progressivement les erreurs (anneaux rouges clignotants).
> 3. **Convergence optimale (Époque 27)** : La frontière atteint un hyperplan séparateur parfait ($x_2 \approx 0.05 x_1 + 2.94$) avec **0 erreur**, déclenchant l'arrêt anticipé de l'algorithme.
>
> *Une version vidéo MP4 Haute Définition (1080p, 24 fps) est également disponible dans le dossier [`pictures/perceptron_apprentissage.mp4`](pictures/perceptron_apprentissage.mp4).*


---
## 8 · Synthèse Générale & Aide-Mémoire (Cheat Sheet)

Félicitations ! Vous avez conçu, entraîné et validé votre premier modèle de Machine Learning en pur NumPy.

### 📋 Résumé des Formules & Syntaxes Clés

| Étape | Formule Mathématique | Syntax NumPy Vectorisée |
|---|---|---|
| **Combinaison Linéaire** | $z = \mathbf{w} \cdot \mathbf{x} + b$ | `z = X @ w + b` |
| **Activation Heaviside** | $\hat{y} = 1 \text{ si } z \ge 0 \text{ sinon } 0$ | `np.where(z >= 0, 1, 0)` |
| **Erreur d'échantillon** | $e_i = y_i - \hat{y}_i$ | `delta = target - pred` |
| **Mise à jour des poids** | $\mathbf{w} \leftarrow \mathbf{w} + \eta \cdot e_i \cdot \mathbf{x}_i$ | `w += lr * delta * x_i` |
| **Mise à jour du biais** | $b \leftarrow b + \eta \cdot e_i$ | `b += lr * delta` |
| **Frontière 2D ($z=0$)** | $x_2 = -\frac{w_1}{w_2} x_1 - \frac{b}{w_2}$ | `x2 = -(w[0]*x1 + b) / w[1]` |
| **Taux de Précision** | $\frac{1}{N} \sum \mathbb{I}(y = \hat{y})$ | `np.mean(y_pred == y)` |

---

### 🚀 De 1957 à Aujourd'hui : La Passerelle vers le Deep Learning Moderne

Le Perceptron simple a des limites (la séparabilité linéaire), mais il contient en germe toute l'architecture des réseaux de neurones actuels :

1. **Remplacer le seuil binaire par une courbe douce** :  
   En remplaçant l'échelon brutal $f(z)$ par la fonction **Sigmoïde** $\sigma(z) = \frac{1}{1 + e^{-z}}$, on obtient des probabilités continues entre $0$ et $1$ → c'est la **Régression Logistique** !
2. **Le Gradient Descendant** :  
   Au lieu de la règle heurisitique de Rosenblatt, on définit une fonction de coût différentiable (comme l'entropie croisée binaire) que l'on optimise par **descente de gradient**.
3. **Le Perceptron Multicouche (MLP)** :  
   En empilant plusieurs neurones en couches cachées successives reliées par des fonctions d'activation non-linéaires (ReLU, GELU), les réseaux sont capables de découper l'espace de manière arbitrairement complexe et de résoudre le XOR, la reconnaissance d'images ou le traitement du langage naturel !
